# 01 — Autoencoder convolucional supervisado para MAG

**Objetivo del notebook:** construir y comprender la arquitectura antes de entrenar con MAVEN.

Este es un **prototipo con señales artificiales**: no consulta BigQuery, no guarda mediciones ni pesos y no representa resultados de detección de MPB.

A futuro el modelo recibirá ventanas de MAG de 600 s, con los canales (B_x), (B_y), (B_z) y (|B|), normalizadas utilizando **exclusivamente estadísticas del fold de entrenamiento**.

Cada ventana tendrá dos objetivos: (1) reconstruir sus mediciones y (2) predecir la cercanía temporal entre el **centro** de la ventana y un cruce MPB conocido.

## 1. Dependencias y convenciones

Trabajamos con tensores **(batch, canales, tiempo)**. No interpolamos muestras para compartir un eje temporal con SWEA.

La máscara `time_mask` indica qué posiciones temporales corresponden a mediciones reales y cuáles son padding. El eje `rel_time_s` permite identificar el instante central físicamente, incluso si cambia la cantidad de muestras.

In [ ]:
import sys
from importlib import import_module
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn

repo_root = Path.cwd()
if not (repo_root / "scripts").is_dir():
    repo_root = repo_root.parent
if not (repo_root / "scripts").is_dir():
    raise RuntimeError("Abrí Jupyter desde la raíz del repositorio o la carpeta notebooks/")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

batching = import_module("scripts.4_preprocessing.4a_native_batches")
torch.manual_seed(7)
np.random.seed(7)
print("PyTorch:", torch.__version__)

## 2. Lote artificial de MAG

Generamos cuatro señales con variaciones suaves y una transición localizada en tiempos diferentes. La transición **no intenta reproducir un fenómeno físico específico**, solamente nos permite comprobar dimensiones, máscaras y etiquetas.

Para el objetivo supervisado usamos

[
y(t_c)=expleft[-rac{(t_c-t_{mathrm{MPB}})^2}{2sigma^2}ight],qquad sigma=60 {m s}.
]

Los ejemplos se centran en (t_c=0); desplazamos el evento sintético con respecto al centro.

**Importante:** la estandarización del ejemplo es sólo artificial. En MAVEN usaremos los parámetros `mean` y `scale` obtenidos con el fold de entrenamiento, nunca valores calculados sobre ventanas de validación.

In [ ]:
t = np.arange(-300.0, 301.0, 1.0)   # 601 posiciones, sólo para esta prueba
offsets_mpb_s = np.array([0.0, 40.0, 120.0, 240.0])
raw_signals = []

for offset in offsets_mpb_s:
    transition = np.tanh((t - offset) / 16.0)
    bx = 6 + 1.5 * np.sin(t / 57) + 2.2 * transition
    by = -1 + 0.9 * np.cos(t / 40) - 0.6 * transition
    bz = 3 + 0.8 * np.sin(t / 25) + 0.4 * transition
    btot = np.sqrt(bx**2 + by**2 + bz**2)
    raw_signals.append(np.stack([bx, by, bz, btot]))

raw = np.stack(raw_signals).astype(np.float32)  # (4, 4, 601)
demo_mean = raw.mean(axis=(0, 2), keepdims=True)
demo_scale = raw.std(axis=(0, 2), keepdims=True)
x_np = (raw - demo_mean) / np.maximum(demo_scale, 1e-6)

# Simulamos longitudes nativas variables: el padding NO cuenta como señal.
valid_lengths = [601, 585, 593, 601]
time_mask_np = np.arange(len(t))[None, :] < np.array(valid_lengths)[:, None]
x_np = np.where(time_mask_np[:, None, :], x_np, 0.0)

x = torch.tensor(x_np)
time_mask = torch.tensor(time_mask_np)
relative_time_s = torch.tensor(np.broadcast_to(t, (len(x), len(t))).copy(),
                               dtype=torch.float32)
y = torch.tensor(np.exp(-0.5 * (offsets_mpb_s / 60.0) ** 2)[:, None],
                 dtype=torch.float32)

batch = batching.NativeBatch(
    x=x,
    time_mask=time_mask,
    rel_time_s=relative_time_s,
    y=y,
    center_s=torch.zeros(len(x), dtype=torch.float64),
    label_offset_s=torch.tensor(-offsets_mpb_s, dtype=torch.float64),
)

print("MAG (B,4,T):", tuple(batch.x.shape))
print("Máscara (B,T):", tuple(batch.time_mask.shape))
print("Objetivo (B,1):", tuple(batch.y.shape))
print("Etiquetas:", batch.y.flatten().numpy().round(3))
print("Mediciones válidas por ventana:", batch.time_mask.sum(dim=1).tolist())

## 3. Arquitectura inicial (prototipo, sujeta a comparación)

El **encoder** aprende una representación latente de 2 canales por instante. Para esta primera versión conserva la resolución temporal, pero comprime los 4 canales MAG a 2 canales latentes.

Las convoluciones dilatadas permiten incorporar información de una región temporal más amplia, no solamente de la muestra central.

La salida latente se conecta a dos ramas:

- **Decoder:** reconstruye los cuatro canales de MAG.
- **Detector:** combina (a) la representación cerca del centro de la ventana y (b) un promedio temporal ponderado alrededor del centro, y devuelve un puntaje de 0 a 1.

La ponderación usa el **tiempo físico relativo** y respeta la máscara de padding. Este diseño inicial no está optimizado ni validado todavía y puede modificarse según los resultados.

In [ ]:
class MagSupervisedAutoencoder(nn.Module):
    def __init__(self, channels=4, hidden=16, latent_channels=2, tau_s=90.0):
        super().__init__()
        self.tau_s = tau_s

        encoder_layers = [nn.Conv1d(channels, hidden, kernel_size=7, padding=3),
                          nn.GELU()]
        for dilation in (1, 2, 4, 8, 16, 32):
            encoder_layers.extend([
                nn.Conv1d(hidden, hidden, kernel_size=5,
                          padding=2 * dilation, dilation=dilation),
                nn.GELU(),
            ])
        encoder_layers.append(nn.Conv1d(hidden, latent_channels, kernel_size=1))
        self.encoder = nn.Sequential(*encoder_layers)

        self.decoder = nn.Sequential(
            nn.Conv1d(latent_channels, hidden, kernel_size=5, padding=2),
            nn.GELU(),
            nn.Conv1d(hidden, channels, kernel_size=5, padding=2),
        )

        self.detector = nn.Sequential(
            nn.Linear(2 * latent_channels, hidden),
            nn.GELU(),
            nn.Linear(hidden, 1),
            nn.Sigmoid(),
        )

    def forward(self, x, time_mask, rel_time_s):
        if x.ndim != 3 or x.shape[1] != 4:
            raise ValueError("Se espera MAG (B,4,T).")
        if time_mask.shape != (x.shape[0], x.shape[-1]):
            raise ValueError("La máscara debe tener forma (B,T).")
        if rel_time_s.shape != time_mask.shape:
            raise ValueError("Los tiempos relativos deben tener forma (B,T).")
        if not time_mask.any(dim=1).all():
            raise ValueError("No se permiten ventanas sin muestras válidas.")

        mask = time_mask[:, None, :].to(x.dtype)
        z = self.encoder(x * mask) * mask
        x_hat = self.decoder(z) * mask

        # Representación de la medición cuyo tiempo está más cerca de t_c=0.
        nearest_center = rel_time_s.abs().masked_fill(~time_mask, float("inf"))
        center_index = nearest_center.argmin(dim=1)
        z_center = z.gather(
            dim=2,
            index=center_index[:, None, None].expand(-1, z.shape[1], 1),
        ).squeeze(-1)

        # Integración local ponderada en segundos físicos, sin usar padding.
        weights = torch.exp(-0.5 * (rel_time_s / self.tau_s) ** 2)
        weights = weights * time_mask.to(weights.dtype)
        z_local = (
            (z * weights[:, None, :]).sum(dim=-1)
            / weights.sum(dim=-1).clamp_min(1e-9)[:, None]
        )

        y_hat = self.detector(torch.cat([z_center, z_local], dim=1))
        return z, x_hat, y_hat

model = MagSupervisedAutoencoder()
print(model)

## 4. Verificar las dimensiones (sin entrenar)

In [ ]:
with torch.no_grad():
    z, x_hat, y_hat = model(batch.x, batch.time_mask, batch.rel_time_s)

print("Entrada:       ", tuple(batch.x.shape))
print("Latente:       ", tuple(z.shape))
print("Reconstrucción:", tuple(x_hat.shape))
print("Detección:     ", tuple(y_hat.shape))
print("Predicciones de una red NO entrenada:", y_hat.flatten().numpy().round(3))
assert x_hat.shape == batch.x.shape
assert y_hat.shape == batch.y.shape
assert bool(torch.all((y_hat >= 0) & (y_hat <= 1)))
assert bool(torch.all(x_hat.masked_select(~batch.time_mask[:, None, :]) == 0))

## 5. Funciones de pérdida y retropropagación

La pérdida combina:

[
mathcal L = mathcal L_{m rec}+lambda_{m det}mathcal L_{m det}.
]

- `L_rec`: error cuadrático de reconstrucción **solo en mediciones válidas**.
- `L_det`: error cuadrático del puntaje con ponderación de los ejemplos cercanos a una etiqueta conocida.
- `lambda_det`: parámetro por ajustar. En este ejemplo usamos 1, **sin afirmar que sea óptimo**.

Esto no convierte los ejemplos alejados de las etiquetas en negativos físicos verificados.

In [ ]:
model.train()
z, x_hat, y_hat = model(batch.x, batch.time_mask, batch.rel_time_s)
loss_rec = batching.masked_reconstruction_mse(x_hat, batch)
loss_det = batching.weighted_proximity_mse(
    y_hat, batch, near_label_weight=10.0
)
lambda_det = 1.0   # sólo para comprobar el cálculo
loss = loss_rec + lambda_det * loss_det

model.zero_grad(set_to_none=True)
loss.backward()
n_parameters = sum(param.numel() for param in model.parameters())
gradients_ok = all(
    p.grad is None or bool(torch.isfinite(p.grad).all())
    for p in model.parameters()
)

print(f"Parámetros entrenables: {n_parameters:,}")
print(f"L_rec={loss_rec.item():.5f}, L_det={loss_det.item():.5f}")
print(f"L_total={loss.item():.5f}; gradientes finitos={gradients_ok}")
assert gradients_ok

## 6. Visualización de la prueba

**Estas curvas son artificiales y el modelo todavía NO está entrenado.** La figura permite revisar qué va a graficar más adelante nuestro análisis: reconstrucción de MAG y predicción frente a la etiqueta gaussiana.

In [ ]:
model.eval()
with torch.no_grad():
    _, reconstruction, prediction = model(
        batch.x, batch.time_mask, batch.rel_time_s
    )

fig, ax = plt.subplots(2, 1, figsize=(10, 7))
idx = 0
ax[0].plot(t, batch.x[idx, 3], label="|B| normalizado (artificial)")
ax[0].plot(t, reconstruction[idx, 3], "--",
           label="Reconstrucción: red sin entrenar")
ax[0].set(xlabel="Tiempo relativo al centro (s)", ylabel="Valor normalizado")
ax[0].legend()
ax[0].grid(alpha=0.2)

grid = np.linspace(0, 300, 300)
target_curve = np.exp(-0.5 * (grid / 60.0) ** 2)
ax[1].plot(grid, target_curve, label="Etiqueta gaussiana objetivo")
ax[1].scatter(offsets_mpb_s, prediction[:, 0].numpy(),
              label="Predicción: red sin entrenar", zorder=3)
ax[1].set(xlabel="Distancia del cruce al centro (s)", ylabel="Puntaje",
          ylim=(-0.05, 1.05))
ax[1].legend()
ax[1].grid(alpha=0.2)
fig.tight_layout()
plt.show()

## 7. Qué falta antes de entrenar con MAVEN

1. Conectar este modelo con las ventanas reales que ya produce `3a_windowing.py` y con los lotes de `4a_native_batches.py`.
2. Cargar para cada fold los valores **globales** de `mean` y `scale` del JSON final generado por `4c` (no del checkpoint).
3. Confirmar que la arquitectura maneje padding correctamente durante el entrenamiento y la evaluación; comparar alternativas y revisar efectos de borde.
4. Diseñar el muestreo de ventanas para que miles de ventanas casi idénticas alrededor de un mismo cruce no dominen el aprendizaje.
5. Entrenar con los meses del conjunto TRAIN, sin ajustar hiperparámetros sobre el conjunto de validación final.
6. Comparar rendimiento del autoencoder supervisado con el de una CNN supervisada sin decoder.
7. Medir **eventos detectados** (precisión, recall, F1 y error temporal), no solamente el error de ventanas correlacionadas.

**Conclusión:** este notebook sólo verifica y documenta un primer diseño de MAG. El modelo real y sus resultados todavía están pendientes.